<a href="https://colab.research.google.com/github/azhgh22/Monet_Style_Generator/blob/main/notebooks/eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Set Env**

In [1]:
%%capture
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
user_name = userdata.get('GITHUB_USERNAME')
mail = userdata.get('GITHUB_MAIL')

!git config --global user.name "{user_name}"
!git config --global user.email "{mail}"
!git clone https://{token}@github.com/azhgh22/Monet_Style_Generator.git
!pip install -r ./Monet_Style_Generator/requirements.txt

# **Imports**

In [2]:
import wandb
import sys
import os
import torch

# Add the root directory of the cloned repository to the Python path
sys.path.append('/content/Monet_Style_Generator')

import importlib
import utils.custom_dataset as custom_dataset_module
import utils.train_test_split as tts_module
import utils.dataset_assembly as ds_assembly_module
import utils.weight_initializer as weight_initializer
import eval.eval_utils as eval_module
import models.GAN.generator_models.resnet_gen_cut as res_gen_cut_module
import models.GAN.cut as cut_module
importlib.reload(custom_dataset_module)
importlib.reload(tts_module)
importlib.reload(ds_assembly_module)
importlib.reload(weight_initializer)
importlib.reload(eval_module)
importlib.reload(res_gen_cut_module)
importlib.reload(cut_module)
from utils.custom_dataset import CustomDataset
from utils.dataset_assembly import DatasetAssembly
from utils.train_test_split import TrainTestSplit
from utils.weight_initializer import WeightsInitializer
from pathlib import Path
from torch.utils.data import DataLoader
import torchvision.transforms as T
from eval.eval_utils import evaluate_mifid
from models.GAN.generator_models.resnet_gen_cut import ResnetGeneratorCut
from models.GAN.cut import Cut



# imort model

import models.GAN.generator_models.resnet_gen as res_gen
import models.GAN.discriminator_models.patchgan as patchgan
import models.GAN.cycle_gan as cycle_gan
import utils.train as train_module
import utils.checkpointer as checkpointer_module
importlib.reload(res_gen)
importlib.reload(patchgan)
importlib.reload(cycle_gan)
importlib.reload(train_module)
importlib.reload(checkpointer_module)
from models.GAN.generator_models.resnet_gen import ResnetGenerator
from models.GAN.discriminator_models.patchgan import PatchGANDiscriminator
from models.GAN.cycle_gan import CycleGAN
from utils.train import Train
from utils.checkpointer import Checkpointer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# **Read Data and Split data**

In [3]:
train, val, test = TrainTestSplit("/content/Monet_Style_Generator/data/photo_jpg/").split(0.7,0.15,0.15)

In [4]:
monet_pictures = list(Path("/content/Monet_Style_Generator/data/monet_jpg/").glob('*.jpg'))

In [5]:
# create datasets
transform = T.Compose([
    T.ToTensor(),
    T.ConvertImageDtype(torch.float)
])

train_part = CustomDataset(train,size=1000)
val_dataset = CustomDataset(val)
test_dataset = CustomDataset(test)

monet_dataset = CustomDataset(monet_pictures)

# train_dataset = DatasetAssembly(train_part, monet_dataset)
# train_dataset = CustomDataset(train_part)

In [6]:

import matplotlib.pyplot as plt
import torch
import torchvision.transforms as transforms
from torchvision.utils import make_grid

def show_img(x: torch.Tensor):
  x = x.detach().cpu()
  x = x.permute(1, 2, 0)

  plt.imshow(x)
  plt.axis("off")
  plt.show()

# **Data Loaders**

In [7]:
train_loader = DataLoader(
    train_part,
    batch_size=1,
    shuffle=False,
    num_workers=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
  )

monet_loader = DataLoader(
    monet_dataset,
    batch_size=1,
    shuffle=False,
)



# **Login Wandb**

In [8]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: azhgh22 (MLBeasts) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# **Eval Cycel_Gan**

In [ ]:
%%capture
monet_generator = ResnetGenerator()
monet_discriminator = PatchGANDiscriminator()

picture_generator = ResnetGenerator()
picture_discriminator = PatchGANDiscriminator()

cycle_gan_model = CycleGAN(monet_generator, picture_generator, monet_discriminator, picture_discriminator).to(device)


checkpoint_dir = "/content/drive/MyDrive/checkpoints/cycle_gan_resnet_v1"

checkpointer = Checkpointer(checkpoint_dir,"cycle_gan",1,False)
train = Train(cycle_gan_model, 30, train_loader, checkpointer, device)
# train.load_checkpoint()

In [ ]:
epochs = 30
CONFIG = {
    "epochs": 30,
    "batch_size": 1,
    "learning_rate": 0.0002,
    "optimizer_beta1": 0.5,
    "optimizer_beta2": 0.999,
    "img_pool_size" : 50,
    "lambda_cycle" : 10,
    "lambda_identity" : 5,
    "GAN loss" : "MSELoss",
    "cycle loss" : "L1Loss",
    "identity loss" : "L1Loss"
}

wandb.init(project="Monet_Generator", entity="azhgh22-free-university-of-tbilisi-", name="CycleGan", config=CONFIG)


for idx in range(5):
      # Ensure generator is in eval mode for consistent inference
      monet_generator.eval()

      train_img_orig = train_part[idx].to(device) # Still need original to generate generated image
      val_img_orig = val_dataset[idx].to(device)

      # Generate images (add batch dimension for generator, then remove for logging)

      print(f"Logging images for epoch {0}, idx {idx}") # Debugging print
      wandb.log({
          f"Generated Samples/Train Generated {idx}": wandb.Image(train_img_orig, caption=f"Epoch {0} Train Generated {idx}"),
          f"Generated Samples/Val Generated {idx}": wandb.Image(val_img_orig, caption=f"Epoch {0} Val Generated {idx}"),
      }, step=0)



try:
  for i in range(1,30+1):
    print("Epoch ",i)
    train.load_checkpoint(i)
    losses = train.epoch_losses
    epoch = i
    generators_loss = losses[-1]["G"]
    mone_disc_loss = losses[-1]["D_A"]
    picture_disc_loss = losses[-1]["D_B"]
    cycle_loss_pict = losses[-1]["cycle_A"]
    cycle_loss_monet = losses[-1]["cycle_B"]
    identity_loss_pict = losses[-1]["idt_A"]
    identity_loss_monet = losses[-1]["idt_B"]

    evaluation_train = evaluate_mifid(monet_generator,train_loader,monet_loader,device,0.5)
    evaluation_val = evaluate_mifid(monet_generator,val_loader,monet_loader,device,0.5)
    fid_train = evaluation_train["FID"]
    MiFID_train = evaluation_train["MiFID"]

    fid_val = evaluation_val["FID"]
    MiFID_val = evaluation_val["MiFID"]

    # Consolidate all scalar metric logging into a single wandb.log call
    metrics_to_log = {
        "Generator Loss": generators_loss,
        "Monet Discriminator Loss": mone_disc_loss,
        "Picture Discriminator Loss": picture_disc_loss,
        "Cycle Loss Picture": cycle_loss_pict,
        "Cycle Loss Monet": cycle_loss_monet,
        "Identity Loss Picture": identity_loss_pict,
        "Identity Loss Monet": identity_loss_monet,
        "FID_train": fid_train,
        "FID_val": fid_val,
        "MiFID_train": MiFID_train,
        "MiFID_val": MiFID_val,
    }
    print(f"Logging metrics for epoch {epoch}: {metrics_to_log}") # Debugging print
    wandb.log(metrics_to_log, step=epoch)

    # Log generated images
    for idx in range(5):
      # Ensure generator is in eval mode for consistent inference
      monet_generator.eval()

      train_img_orig = train_part[idx].to(device) # Still need original to generate generated image
      val_img_orig = val_dataset[idx].to(device)

      # Generate images (add batch dimension for generator, then remove for logging)
      with torch.no_grad():
        train_gen_img = monet_generator(train_img_orig.unsqueeze(0)).squeeze(0).cpu()
        val_gen_img = monet_generator(val_img_orig.unsqueeze(0)).squeeze(0).cpu()

      print(f"Logging images for epoch {epoch}, idx {idx}") # Debugging print
      wandb.log({
          f"Generated Samples/Train Generated {idx}": wandb.Image(train_gen_img, caption=f"Epoch {epoch} Train Generated {idx}"),
          f"Generated Samples/Val Generated {idx}": wandb.Image(val_gen_img, caption=f"Epoch {epoch} Val Generated {idx}"),
      }, step=epoch)

    # Get state dictionaries
    monet_syle_gen_state = monet_generator.state_dict()
    picture_gen_state = picture_generator.state_dict()
    monet_disc_state = monet_discriminator.state_dict()
    picture_disc_state = picture_discriminator.state_dict()

    # Save and log monet_generator state dict as a model artifact
    torch.save(monet_syle_gen_state, f"monet_generator_epoch_{epoch}.pt")
    monet_gen_artifact = wandb.Artifact(f"monet_generator", type="model")
    monet_gen_artifact.add_file(f"monet_generator_epoch_{epoch}.pt")
    print(f"Logging monet_generator artifact for epoch {epoch}") # Debugging print
    wandb.log_artifact(monet_gen_artifact, aliases=["latest", f"epoch_{epoch}"])
    os.remove(f"monet_generator_epoch_{epoch}.pt") # Clean up local file

    # Save and log picture_generator state dict as a model artifact
    torch.save(picture_gen_state, f"picture_generator_epoch_{epoch}.pt")
    picture_gen_artifact = wandb.Artifact(f"picture_generator", type="model")
    picture_gen_artifact.add_file(f"picture_generator_epoch_{epoch}.pt")
    print(f"Logging picture_generator artifact for epoch {epoch}") # Debugging print
    wandb.log_artifact(picture_gen_artifact, aliases=["latest", f"epoch_{epoch}"])
    os.remove(f"picture_generator_epoch_{epoch}.pt") # Clean up local file

    # Save and log monet_discriminator state dict as a model artifact
    torch.save(monet_disc_state, f"monet_discriminator_epoch_{epoch}.pt")
    monet_disc_artifact = wandb.Artifact(f"monet_discriminator", type="model")
    monet_disc_artifact.add_file(f"monet_discriminator_epoch_{epoch}.pt")
    print(f"Logging monet_discriminator artifact for epoch {epoch}") # Debugging print
    wandb.log_artifact(monet_disc_artifact, aliases=["latest", f"epoch_{epoch}"])
    os.remove(f"monet_discriminator_epoch_{epoch}.pt") # Clean up local file

    # Save and log picture_discriminator state dict as a model artifact
    torch.save(picture_disc_state, f"picture_discriminator_epoch_{epoch}.pt")
    picture_disc_artifact = wandb.Artifact(f"picture_discriminator", type="model")
    picture_disc_artifact.add_file(f"picture_discriminator_epoch_{epoch}.pt")
    print(f"Logging picture_discriminator artifact for epoch {epoch}") # Debugging print
    wandb.log_artifact(picture_disc_artifact, aliases=["latest", f"epoch_{epoch}"])
    os.remove(f"picture_discriminator_epoch_{epoch}.pt") # Clean up local file

finally:
  print("Ensuring wandb.finish() is called.")
  wandb.finish()


Logging images for epoch 0, idx 0
Logging images for epoch 0, idx 1
Logging images for epoch 0, idx 2
Logging images for epoch 0, idx 3
Logging images for epoch 0, idx 4
Epoch  1
Loaded checkpoint for epoch 1: /content/drive/MyDrive/checkpoints/cycle_gan_resnet_v1/cycle_gan_epoch_1.pt


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Extracting features: 100%|██████████| 300/300 [00:04<00:00, 73.96it/s]


Logging metrics for epoch 1: {'Generator Loss': 4.631078126466328, 'Monet Discriminator Loss': 0.24821814586730978, 'Picture Discriminator Loss': 0.22743102264792103, 'Cycle Loss Picture': 1.3219814459047228, 'Cycle Loss Monet': 1.2931364049308434, 'Identity Loss Picture': 0.5977652353384705, 'Identity Loss Monet': 0.6143855071387251, 'FID_train': 146.52015463281663, 'FID_val': 149.63296030229498, 'MiFID_train': 493.00701520326044, 'MiFID_val': 498.9621445604739}
Logging images for epoch 1, idx 0
Logging images for epoch 1, idx 1
Logging images for epoch 1, idx 2
Logging images for epoch 1, idx 3
Logging images for epoch 1, idx 4
Logging monet_generator artifact for epoch 1
Logging picture_generator artifact for epoch 1
Logging monet_discriminator artifact for epoch 1
Logging picture_discriminator artifact for epoch 1
Epoch  2
Loaded checkpoint for epoch 2: /content/drive/MyDrive/checkpoints/cycle_gan_resnet_v1/cycle_gan_epoch_2.pt


Extracting features: 100%|██████████| 300/300 [00:05<00:00, 59.71it/s]


Logging metrics for epoch 2: {'Generator Loss': 3.963167785606005, 'Monet Discriminator Loss': 0.22028448316221358, 'Picture Discriminator Loss': 0.1375965224628133, 'Cycle Loss Picture': 1.0370828452581706, 'Cycle Loss Monet': 1.0450205436045206, 'Identity Loss Picture': 0.48153927394688445, 'Identity Loss Monet': 0.48783165297515985, 'FID_train': 118.65719880998427, 'FID_val': 119.82868015404391, 'MiFID_train': 441.319725412395, 'MiFID_val': 441.93649399386993}
Logging images for epoch 2, idx 0
Logging images for epoch 2, idx 1
Logging images for epoch 2, idx 2
Logging images for epoch 2, idx 3
Logging images for epoch 2, idx 4
Logging monet_generator artifact for epoch 2
Logging picture_generator artifact for epoch 2
Logging monet_discriminator artifact for epoch 2
Logging picture_discriminator artifact for epoch 2
Epoch  3
Loaded checkpoint for epoch 3: /content/drive/MyDrive/checkpoints/cycle_gan_resnet_v1/cycle_gan_epoch_3.pt


...

Extracting features: 100%|██████████| 300/300 [00:04<00:00, 72.32it/s]


Logging metrics for epoch 29: {'Generator Loss': 2.780783156626504, 'Monet Discriminator Loss': 0.1696739273544339, 'Picture Discriminator Loss': 0.03336400040115298, 'Cycle Loss Picture': 0.5871105452275741, 'Cycle Loss Monet': 0.512870727689998, 'Identity Loss Picture': 0.24359355281419837, 'Identity Loss Monet': 0.2474092066052044, 'FID_train': 93.468835925246, 'FID_val': 94.51677731550083, 'MiFID_train': 362.621343334061, 'MiFID_val': 363.89349989109525}
Logging images for epoch 29, idx 0
Logging images for epoch 29, idx 1
Logging images for epoch 29, idx 2
Logging images for epoch 29, idx 3
Logging images for epoch 29, idx 4
Logging monet_generator artifact for epoch 29
Logging picture_generator artifact for epoch 29
Logging monet_discriminator artifact for epoch 29
Logging picture_discriminator artifact for epoch 29
Epoch  30
Loaded checkpoint for epoch 30: /content/drive/MyDrive/checkpoints/cycle_gan_resnet_v1/cycle_gan_epoch_30.pt


Extracting features: 100%|██████████| 300/300 [00:05<00:00, 58.26it/s]


Logging metrics for epoch 30: {'Generator Loss': 2.7720205517643794, 'Monet Discriminator Loss': 0.17495391269524893, 'Picture Discriminator Loss': 0.032825492528523195, 'Cycle Loss Picture': 0.5840199127841712, 'Cycle Loss Monet': 0.5092852012781026, 'Identity Loss Picture': 0.2401489993853028, 'Identity Loss Monet': 0.24384550829359394, 'FID_train': 93.05843736746144, 'FID_val': 95.0172391608362, 'MiFID_train': 364.3066663258139, 'MiFID_val': 364.73054045080846}
Logging images for epoch 30, idx 0
Logging images for epoch 30, idx 1
Logging images for epoch 30, idx 2
Logging images for epoch 30, idx 3
Logging images for epoch 30, idx 4
Logging monet_generator artifact for epoch 30
Logging picture_generator artifact for epoch 30
Logging monet_discriminator artifact for epoch 30
Logging picture_discriminator artifact for epoch 30
Ensuring wandb.finish() is called.


Cycle Loss Monet,█▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
Cycle Loss Picture,█▅▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
FID_train,█▄▃▂▂▃▂▂▂▂▂▁▂▂▂▂▂▂▂▁▂▂▁▁▃▂▁▁▁▁
FID_val,█▄▃▂▂▃▂▂▂▂▁▁▂▂▂▂▂▂▂▁▂▂▁▁▃▂▁▁▁▁
Generator Loss,█▅▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁
Identity Loss Monet,█▆▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁
Identity Loss Picture,█▆▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
MiFID_train,█▅▄▃▂▄▃▂▂▂▂▁▂▂▂▂▂▂▂▁▂▂▁▁▃▂▁▁▁▁
MiFID_val,█▅▃▃▂▄▃▂▂▂▂▁▂▂▂▂▂▂▂▁▁▂▁▁▃▂▁▁▁▁
Monet Discriminator Loss,█▆▅▅▄▄▃▂▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃
+1,...


# **Evaluate Cut resnet**

In [9]:
%%capture
cut_monet_generator = ResnetGeneratorCut()
cut_monet_discriminator = PatchGANDiscriminator()

cut_model = Cut(cut_monet_generator, cut_monet_discriminator)
cut_model.to(device)

checkpoint_dir = "/content/drive/MyDrive/checkpoints/cut_resnet_v2"

checkpointer = Checkpointer(checkpoint_dir,"cut",1,False)
train = Train(cut_model, 30, train_loader, checkpointer, device)

In [16]:
epochs = 30
CONFIG = {
    "epochs": 30,
    "batch_size": 1,
    "learning_rate": 0.0002,
    "optimizer_beta1": 0.5,
    "optimizer_beta2": 0.999,
    "img_pool_size" : 50,
    "lambda_cycle" : 1,
    "lambda_identity" : 0.5,
    "GAN loss" : "MSELoss",
    "nce loss" : "CrossEntropyLoss",
    "identity loss" : "L1Loss",
    "nce layers" : [0,1,2,3,4],
    "generator" : "Resnet",
    "disciminator" : "PatchGAN",
    "MLP head" : "Linear -> relu -> linear",
}

wandb.init(project="Monet_Generator", entity="azhgh22-free-university-of-tbilisi-", name="Cut_Resnet", config=CONFIG)


for idx in range(5):
      # Ensure generator is in eval mode for consistent inference
      cut_monet_generator.eval()

      train_img_orig = train_part[idx].to(device) # Still need original to generate generated image
      val_img_orig = val_dataset[idx].to(device)

      # Generate images (add batch dimension for generator, then remove for logging)

      print(f"Logging images for epoch {0}, idx {idx}") # Debugging print
      wandb.log({
          f"Generated Samples/Train Generated {idx}": wandb.Image(train_img_orig, caption=f"Epoch {0} Train Generated {idx}"),
          f"Generated Samples/Val Generated {idx}": wandb.Image(val_img_orig, caption=f"Epoch {0} Val Generated {idx}"),
      }, step=0)



try:
  for i in range(1,30+1):
    print("Epoch ",i)
    train.load_checkpoint(i)
    losses = train.epoch_losses
    epoch = i
    generators_loss = losses[-1]["G"]
    disc_loss = losses[-1]["D"]
    Gan = losses[-1]["GAN"]
    patchNCE_loss = losses[-1]["PatchNCE"]
    identity_loss = losses[-1]["Identity"]

    evaluation_train = evaluate_mifid(cut_monet_generator,train_loader,monet_loader,device,0.5)
    evaluation_val = evaluate_mifid(cut_monet_generator,val_loader,monet_loader,device,0.5)
    fid_train = evaluation_train["FID"]
    MiFID_train = evaluation_train["MiFID"]

    fid_val = evaluation_val["FID"]
    MiFID_val = evaluation_val["MiFID"]

    # Consolidate all scalar metric logging into a single wandb.log call
    metrics_to_log = {
        "Generator Loss": generators_loss,
        "Discriminator Loss": disc_loss,
        "GAN Loss": Gan,
        "PatchNCE Loss": patchNCE_loss,
        "Identity Loss": identity_loss,
        "FID_train": fid_train,
        "FID_val": fid_val,
        "MiFID_train": MiFID_train,
        "MiFID_val": MiFID_val,
    }
    print(f"Logging metrics for epoch {epoch}: {metrics_to_log}") # Debugging print
    wandb.log(metrics_to_log, step=epoch)

    # Log generated images
    for idx in range(5):
      # Ensure generator is in eval mode for consistent inference
      cut_monet_generator.eval()

      train_img_orig = train_part[idx].to(device) # Still need original to generate generated image
      val_img_orig = val_dataset[idx].to(device)

      # Generate images (add batch dimension for generator, then remove for logging)
      with torch.no_grad():
        train_gen_img = cut_monet_generator(train_img_orig.unsqueeze(0)).squeeze(0).cpu()
        val_gen_img = cut_monet_generator(val_img_orig.unsqueeze(0)).squeeze(0).cpu()

      print(f"Logging images for epoch {epoch}, idx {idx}") # Debugging print
      wandb.log({
          f"Generated Samples/Train Generated {idx}": wandb.Image(train_gen_img, caption=f"Epoch {epoch} Train Generated {idx}"),
          f"Generated Samples/Val Generated {idx}": wandb.Image(val_gen_img, caption=f"Epoch {epoch} Val Generated {idx}"),
      }, step=epoch)

    # Get state dictionaries
    mone_gen = cut_monet_generator.state_dict()
    mone_disc = cut_monet_discriminator.state_dict()
    cut_mlp = cut_model.mlps.state_dict()

    # Save and log cut_monet_generator state dict as a model artifact
    torch.save(mone_gen, f"cut_monet_generator_epoch_{epoch}.pt")
    monet_gen_artifact = wandb.Artifact(f"cut_monet_generator", type="model")
    monet_gen_artifact.add_file(f"cut_monet_generator_epoch_{epoch}.pt")
    print(f"Logging cut_monet_generator artifact for epoch {epoch}")
    wandb.log_artifact(monet_gen_artifact, aliases=["latest", f"epoch_{epoch}"])
    os.remove(f"cut_monet_generator_epoch_{epoch}.pt")

    # Save and log cut_monet_discriminator state dict as a model artifact
    torch.save(mone_disc, f"cut_monet_discriminator_epoch_{epoch}.pt")
    monet_disc_artifact = wandb.Artifact(f"cut_monet_discriminator", type="model")
    monet_disc_artifact.add_file(f"cut_monet_discriminator_epoch_{epoch}.pt")
    print(f"Logging cut_monet_discriminator artifact for epoch {epoch}")
    wandb.log_artifact(monet_disc_artifact, aliases=["latest", f"epoch_{epoch}"])
    os.remove(f"cut_monet_discriminator_epoch_{epoch}.pt")

    # Save and log cut_mlp state dict as a model artifact
    torch.save(cut_mlp, f"cut_mlp_epoch_{epoch}.pt")
    cut_mlp_artifact = wandb.Artifact(f"cut_mlp", type="model")
    cut_mlp_artifact.add_file(f"cut_mlp_epoch_{epoch}.pt")
    print(f"Logging cut_mlp artifact for epoch {epoch}")
    wandb.log_artifact(cut_mlp_artifact, aliases=["latest", f"epoch_{epoch}"])
    os.remove(f"cut_mlp_epoch_{epoch}.pt")


finally:
  print("Ensuring wandb.finish() is called.")
  wandb.finish()

Logging images for epoch 0, idx 0
Logging images for epoch 0, idx 1
Logging images for epoch 0, idx 2
Logging images for epoch 0, idx 3
Logging images for epoch 0, idx 4
Epoch  1
Loaded checkpoint for epoch 1: /content/drive/MyDrive/checkpoints/cut_resnet_v2/cut_epoch_1.pt


Extracting features: 100%|██████████| 300/300 [00:04<00:00, 74.91it/s]


Logging metrics for epoch 1: {'Generator Loss': 1.6882082108700125, 'Discriminator Loss': 0.27948686359975816, 'GAN Loss': 0.36354927132091436, 'PatchNCE Loss': 1.258014817246082, 'Identity Loss': 0.1332882447837513, 'FID_train': 103.35420495506636, 'FID_val': 104.79826887114069, 'MiFID_train': 389.2512618190164, 'MiFID_val': 391.3094231460001}
Logging images for epoch 1, idx 0
Logging images for epoch 1, idx 1
Logging images for epoch 1, idx 2
Logging images for epoch 1, idx 3
Logging images for epoch 1, idx 4
Logging cut_monet_generator artifact for epoch 1
Logging cut_monet_discriminator artifact for epoch 1
Logging cut_mlp artifact for epoch 1
Epoch  2
Loaded checkpoint for epoch 2: /content/drive/MyDrive/checkpoints/cut_resnet_v2/cut_epoch_2.pt


Extracting features: 100%|██████████| 300/300 [00:04<00:00, 71.51it/s]


Logging metrics for epoch 2: {'Generator Loss': 1.2755661008927186, 'Discriminator Loss': 0.2623999448061304, 'GAN Loss': 0.31933933013004234, 'PatchNCE Loss': 0.90144154787596, 'Identity Loss': 0.10957044895876, 'FID_train': 102.55138171596886, 'FID_val': 104.25264527114791, 'MiFID_train': 387.1230554211904, 'MiFID_val': 390.9655760004969}
Logging images for epoch 2, idx 0
Logging images for epoch 2, idx 1
Logging images for epoch 2, idx 2
Logging images for epoch 2, idx 3
Logging images for epoch 2, idx 4
Logging cut_monet_generator artifact for epoch 2
Logging cut_monet_discriminator artifact for epoch 2
Logging cut_mlp artifact for epoch 2
Epoch  3
Loaded checkpoint for epoch 3: /content/drive/MyDrive/checkpoints/cut_resnet_v2/cut_epoch_3.pt


...

Extracting features: 100%|██████████| 300/300 [00:04<00:00, 73.35it/s]


Logging metrics for epoch 29: {'Generator Loss': 1.7333306580648837, 'Discriminator Loss': 0.09569459825253528, 'GAN Loss': 0.763342740317452, 'PatchNCE Loss': 0.9180148101971007, 'Identity Loss': 0.10394621554155611, 'FID_train': 102.47311578277522, 'FID_val': 102.67218866025907, 'MiFID_train': 406.36372993208, 'MiFID_val': 405.37103743780443}
Logging images for epoch 29, idx 0
Logging images for epoch 29, idx 1
Logging images for epoch 29, idx 2
Logging images for epoch 29, idx 3
Logging images for epoch 29, idx 4
Logging cut_monet_generator artifact for epoch 29
Logging cut_monet_discriminator artifact for epoch 29
Logging cut_mlp artifact for epoch 29
Epoch  30
Loaded checkpoint for epoch 30: /content/drive/MyDrive/checkpoints/cut_resnet_v2/cut_epoch_30.pt


Extracting features: 100%|██████████| 300/300 [00:04<00:00, 63.01it/s]


Logging metrics for epoch 30: {'Generator Loss': 1.7437398461394324, 'Discriminator Loss': 0.0916124005853779, 'GAN Loss': 0.7735722463969723, 'PatchNCE Loss': 0.9184931468186802, 'Identity Loss': 0.10334890777297345, 'FID_train': 104.33850833662191, 'FID_val': 104.47435249444446, 'MiFID_train': 405.19849468815016, 'MiFID_val': 402.617346091869}
Logging images for epoch 30, idx 0
Logging images for epoch 30, idx 1
Logging images for epoch 30, idx 2
Logging images for epoch 30, idx 3
Logging images for epoch 30, idx 4
Logging cut_monet_generator artifact for epoch 30
Logging cut_monet_discriminator artifact for epoch 30
Logging cut_mlp artifact for epoch 30
Ensuring wandb.finish() is called.


Discriminator Loss,█▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁
FID_train,▂▂▁▁▂▃▁▂▁▃▂▄▃▄▃▂▃▂▃█▃▂▂▂▂▄▂▂▂▂
FID_val,▂▂▁▁▂▃▁▂▁▃▂▃▃▅▃▂▃▂▃█▂▂▂▂▂▃▁▂▂▂
GAN Loss,▂▁▁▁▁▁▂▂▂▂▃▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇████
Generator Loss,▇▂▁▁▁▁▂▂▂▂▃▃▄▄▅▅▆▆▆▇▇▇▇▇▇█████
Identity Loss,█▃▂▁▁▁▁▂▂▂▂▃▂▂▂▂▃▃▃▂▃▃▂▂▂▂▂▂▂▂
MiFID_train,▂▂▁▂▂▃▁▂▁▃▂▄▃▄▃▂▃▂▃█▃▃▃▂▃▄▂▂▂▂
MiFID_val,▂▂▁▁▂▃▁▂▁▃▂▄▃▅▃▂▃▂▃█▃▃▃▂▃▄▂▂▂▂
PatchNCE Loss,█▂▁▁▁▁▁▂▁▁▂▂▂▂▂▂▂▃▃▃▃▂▂▂▂▂▂▂▂▂
Discriminator Loss,0.09161
FID_train,104.33851
